# 06 - Predicción

Generamos el archivo de entrega con la columna `smoking_prediction`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

RAW = Path('..') / 'data' / 'raw'
PROCESSED = Path('..') / 'data' / 'processed'
MODELS = Path('..') / 'models'
PROCESSED.mkdir(exist_ok=True, parents=True)

df_unlabeled = pd.read_excel(RAW / 'smoking_unlabeled.xlsx')
model = joblib.load(MODELS / 'model.joblib')
print('Dataset sin etiquetar :', df_unlabeled.shape)
print('Modelo                :', type(model.named_steps['clf']).__name__)

Dataset sin etiquetar : (5692, 26)
Modelo                : XGBClassifier


## Aplicacion del pipeline

El `Pipeline` adentro ya incluye el `ColumnTransformer` ajustado en entrenamiento, asi que al pasarle el DataFrame con las mismas columnas de feature, todo el preprocesamiento se aplica automaticamente.

In [2]:
DROP_COLS = ['ID', 'oral']  # mismo que entrenamiento
X_pred = df_unlabeled.drop(columns=DROP_COLS)
predicciones = model.predict(X_pred).astype(int)
print('Distribucion predicciones:')
print(pd.Series(predicciones).value_counts().sort_index())

Distribucion predicciones:
0    3485
1    2207
Name: count, dtype: int64


## Armado del archivo final

Mantengo todas las columnas originales del dataset sin etiquetar + la nueva columna `smoking_prediction` con los valores 0/1, tal como pide la consigna.

In [3]:
df_out = df_unlabeled.copy()
df_out['smoking_prediction'] = predicciones
df_out.head()

,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,oral,dental caries,tartar,smoking_prediction
0,27358,M,25,160,65,3.420139,0.045139,0.002778,0.041667,0.041667,...,0.631250,0.041667,0.006250,0.750000,0.708333,0.708333,Y,0,Y,0
1,27364,M,30,180,80,3.458333,0.043056,0.006250,0.041667,0.041667,...,0.587500,0.041667,0.041667,0.791667,1.125000,1.333333,Y,0,N,1
2,27368,M,55,165,60,3.416667,0.004861,0.005556,0.041667,0.041667,...,0.625694,0.041667,0.006250,1.083333,1.291667,2.000000,Y,1,Y,1
3,27378,M,20,175,75,3.625000,0.045139,0.045139,0.041667,0.041667,...,0.627083,0.041667,0.043750,0.833333,0.583333,0.458333,Y,0,N,0
4,27381,M,25,165,80,3.791667,0.043056,0.041667,0.041667,0.041667,...,0.670833,0.041667,0.041667,1.250000,1.625000,1.958333,Y,1,Y,0


### Chequeos finales

In [4]:
# chequeos rápidos antes de exportar
assert len(df_out) == len(df_unlabeled), 'distinta cantidad de filas!'
assert 'smoking_prediction' in df_out.columns, 'falta la columna de prediccion'
assert set(df_out['smoking_prediction'].unique()).issubset({0, 1}), 'valores fuera de {0,1}'
assert df_out['ID'].is_unique, 'IDs duplicados'
print('OK - archivo listo para exportar')

OK - archivo listo para exportar


## Exportacion

In [5]:
out_csv  = PROCESSED / 'predictions.csv'
out_xlsx = PROCESSED / 'predictions.xlsx'
df_out.to_csv(out_csv, index=False)
df_out.to_excel(out_xlsx, index=False)
print('Guardado:')
print(' ', out_csv)
print(' ', out_xlsx)
print()
print(f'Filas: {len(df_out)}  Fumadores predichos: {int(df_out["smoking_prediction"].sum())}'
      f'  ({df_out["smoking_prediction"].mean()*100:.1f}%)')

Guardado:
  ../data/processed/predictions.csv
  ../data/processed/predictions.xlsx

Filas: 5692  Fumadores predichos: 2207  (38.8%)
